<div align="center">

**Caso Práctico: Modelado de PD Cuantitativa**

<img src="https://www.python.org/static/community_logos/python-logo-master-v3-TM.png" alt="Python" width="200"/>

</div>

# Caso Práctico: Modelado de Probabilidad de Incumplimiento (PD) Cuantitativa

## Introducción al Problema

En gestión de riesgo crediticio, la **Probabilidad de Incumplimiento (PD - Probability of Default)** es una métrica fundamental que representa la probabilidad de que un prestatario no cumpla con sus obligaciones de pago.

**Enfoque Tradicional**: La PD se modela como una variable binaria (0 = no incumple, 1 = incumple) usando modelos como Logit o Probit.

**Enfoque Cuantitativo**: En este ejemplo, modelaremos la PD como una **variable continua entre 0 y 1**, lo cual es útil cuando:
- Tenemos calificaciones de riesgo internas que asignan PDs específicas (ej: 0.5%, 1.2%, 3.5%)
- Queremos modelar cómo factores macroeconómicos afectan la PD de manera continua
- Necesitamos predicciones más granulares que simples categorías

### Contexto del Negocio

Eres un analista de riesgo crediticio en un banco. Tienes acceso a:
- **Histórico de PD cuantitativa** de una cartera de préstamos corporativos (últimos 5 años, datos mensuales)
- **Variables macroeconómicas** que pueden afectar la capacidad de pago de las empresas

**Pregunta de Negocio**: ¿Cómo podemos predecir la PD futura basándonos en condiciones macroeconómicas?

---

**Nota**: Este ejemplo sigue la metodología establecida en el notebook principal de Regresión Lineal, aplicando los 5 pasos fundamentales del proceso de modelado.

## Paso 0: Cargar la Base de Datos

Primero, cargamos la base de datos histórica que fue generada previamente. Esta base de datos contiene:
- **PD cuantitativa** (variable dependiente): valores entre 0 y 1
- **Variables macroeconómicas** (variables independientes): 6 variables que pueden afectar la PD

In [1]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from scipy import stats
from scipy.stats import shapiro, jarque_bera, probplot
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Librerías importadas exitosamente")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Cargar la base de datos
# Asegúrate de que el archivo 'datos_pd_historico.csv' esté en la misma carpeta
# o ajusta la ruta según corresponda

import os

# Obtener la ruta del directorio actual
directorio_actual = os.path.dirname(os.path.abspath(''))
ruta_datos = os.path.join(directorio_actual, 'datos_pd_historico.csv')

# Si el archivo no está en el directorio actual, intentar en la carpeta Clase_2
if not os.path.exists(ruta_datos):
    # Intentar ruta relativa desde el notebook
    ruta_datos = 'datos_pd_historico.csv'

try:
    datos_pd = pd.read_csv(ruta_datos, parse_dates=['fecha'])
    print("=" * 80)
    print("BASE DE DATOS CARGADA EXITOSAMENTE")
    print("=" * 80)
    print(f"\nArchivo: {ruta_datos}")
    print(f"Número de observaciones: {len(datos_pd)}")
    print(f"Período: {datos_pd['fecha'].min().strftime('%Y-%m-%d')} a {datos_pd['fecha'].max().strftime('%Y-%m-%d')}")
    print(f"\nVariables en la base de datos:")
    for i, col in enumerate(datos_pd.columns, 1):
        print(f"  {i}. {col}")
except FileNotFoundError:
    print("=" * 80)
    print("ERROR: No se encontró el archivo de datos")
    print("=" * 80)
    print("\nPor favor, ejecuta primero el script 'generar_datos_pd.py' para crear la base de datos.")
    print("\nPuedes ejecutarlo desde la terminal con:")
    print("  python generar_datos_pd.py")
    print("\nO desde Python/Jupyter:")
    print("  exec(open('generar_datos_pd.py').read())")

In [ ]:
# Verificar y mostrar información básica de los datos
print("=" * 80)
print("INFORMACIÓN DE LA BASE DE DATOS")
print("=" * 80)
print("\nPrimeras 5 observaciones:")
print(datos_pd.head().to_string(index=False))
print("\nÚltimas 5 observaciones:")
print(datos_pd.tail().to_string(index=False))
print("\nEstadísticas descriptivas:")
print(datos_pd.describe().round(2))

## Exploración Inicial de los Datos

Antes de construir el modelo, es crucial entender los datos y las relaciones entre variables:

In [ ]:
# Visualización de la serie temporal de PD
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Gráfico 1: Evolución temporal de PD
axes[0].plot(datos_pd['fecha'], datos_pd['pd_porcentaje'], linewidth=2, marker='o', markersize=4)
axes[0].set_xlabel('Fecha', fontsize=12)
axes[0].set_ylabel('PD (%)', fontsize=12)
axes[0].set_title('Evolución Temporal de la Probabilidad de Incumplimiento (PD)', 
                  fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Gráfico 2: Evolución de variables macroeconómicas clave
ax2_twin = axes[1].twinx()
line1 = axes[1].plot(datos_pd['fecha'], datos_pd['desempleo'], 'b-', label='Desempleo (%)', linewidth=2)
line2 = axes[1].plot(datos_pd['fecha'], datos_pd['pib_crecimiento'], 'g-', label='PIB Crecimiento (%)', linewidth=2)
line3 = ax2_twin.plot(datos_pd['fecha'], datos_pd['tasa_interes'], 'r-', label='Tasa Interés (%)', linewidth=2)

# Combinar leyendas
lines = line1 + line2 + line3
labels = [l.get_label() for l in lines]
axes[1].legend(lines, labels, loc='upper left')

axes[1].set_xlabel('Fecha', fontsize=12)
axes[1].set_ylabel('Desempleo y PIB (%)', fontsize=12, color='black')
ax2_twin.set_ylabel('Tasa de Interés (%)', fontsize=12, color='r')
axes[1].set_title('Evolución de Variables Macroeconómicas Clave', 
                  fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación
variables_corr = ['pd_decimal', 'desempleo', 'pib_crecimiento', 'tasa_interes', 
                  'inflacion', 'confianza_empresarial', 'volatilidad_mercado']
corr_matrix = datos_pd[variables_corr].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, square=True, 
            linewidths=1, cbar_kws={"shrink": 0.8}, fmt='.2f')
plt.title('Matriz de Correlación: PD vs Variables Macroeconómicas', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelaciones con PD (de mayor a menor):")
correlaciones_pd = corr_matrix['pd_decimal'].sort_values(ascending=False)
print(correlaciones_pd[1:])  # Excluir correlación consigo misma

## Aplicando la Metodología de los 5 Pasos

Siguiendo la metodología establecida en el notebook principal, vamos a construir el modelo paso a paso:

### Paso 1: Elegir una Clase de Modelo

**Decisión**: Regresión Lineal Múltiple

**Justificación**:
- La PD es una variable continua (0-1)
- Queremos entender la relación lineal entre variables macroeconómicas y PD
- Necesitamos interpretabilidad para decisiones de negocio
- Es un buen punto de partida antes de considerar modelos no lineales

**Nota**: Aunque la PD está entre 0 y 1, usaremos regresión lineal. Si las predicciones salen fuera de [0,1], podemos aplicar transformaciones o usar modelos alternativos (Logit transformado).

In [ ]:
# Paso 1: Elegir clase de modelo
print("=" * 80)
print("PASO 1: ELEGIR CLASE DE MODELO")
print("=" * 80)
print("\nModelo seleccionado: Regresión Lineal Múltiple")
print("Justificación:")
print("  - Variable dependiente continua (PD entre 0 y 1)")
print("  - Múltiples variables independientes (macroeconómicas)")
print("  - Necesidad de interpretabilidad para decisiones de negocio")
print("  - Base sólida antes de considerar modelos más complejos")

### Paso 2: Elegir Hiperparámetros del Modelo

**Configuración**:
- `fit_intercept=True`: Queremos incluir el intercepto (PD base cuando todas las variables macro son cero)
- No normalizamos porque queremos interpretar los coeficientes directamente

In [ ]:
# Paso 2: Configurar hiperparámetros
modelo_pd = LinearRegression(fit_intercept=True)

print("=" * 80)
print("PASO 2: CONFIGURAR HIPERPARÁMETROS")
print("=" * 80)
print(f"\nModelo configurado:")
print(f"  - fit_intercept: {modelo_pd.fit_intercept}")
print(f"  - Normalización: No (queremos interpretabilidad directa)")

### Paso 3: Organizar los Datos

**Preparación**:
- Separar variables independientes (X) y dependiente (y)
- Dividir en conjunto de entrenamiento y prueba (80/20)
- Asegurar formato correcto (matriz 2D para X, vector 1D para y)

In [ ]:
# Paso 3: Organizar los datos
variables_independientes = ['desempleo', 'pib_crecimiento', 'tasa_interes', 
                           'inflacion', 'confianza_empresarial', 'volatilidad_mercado']

X = datos_pd[variables_independientes].values  # Matriz 2D
y = datos_pd['pd_decimal'].values  # Vector 1D

# Dividir en entrenamiento (80%) y prueba (20%)
# Usamos shuffle=False para mantener el orden temporal
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False, random_state=42
)

print("=" * 80)
print("PASO 3: ORGANIZAR LOS DATOS")
print("=" * 80)
print(f"\nForma de los datos:")
print(f"  X (características): {X.shape} - {X.shape[0]} observaciones, {X.shape[1]} variables")
print(f"  y (objetivo): {y.shape} - {len(y)} observaciones")
print(f"\nDivisión entrenamiento/prueba:")
print(f"  Entrenamiento: {X_train.shape[0]} observaciones ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Prueba: {X_test.shape[0]} observaciones ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nVariables independientes incluidas:")
for i, var in enumerate(variables_independientes, 1):
    print(f"  {i}. {var}")

### Paso 4: Ajustar el Modelo a los Datos

Entrenamos el modelo usando el método de Mínimos Cuadrados Ordinarios (OLS).

In [ ]:
# Paso 4: Ajustar el modelo
modelo_pd.fit(X_train, y_train)

print("=" * 80)
print("PASO 4: AJUSTAR EL MODELO")
print("=" * 80)
print("\nModelo entrenado exitosamente usando Mínimos Cuadrados Ordinarios (OLS)")
print("\nCoeficientes del modelo:")
print(f"  Intercepto (β₀): {modelo_pd.intercept_:.6f}")

print("\n  Coeficientes de regresión:")
coeficientes_df = pd.DataFrame({
    'Variable': variables_independientes,
    'Coeficiente (β)': modelo_pd.coef_,
    'Interpretación': [
        f'Cambio en PD por cada 1% de aumento en desempleo',
        f'Cambio en PD por cada 1% de aumento en crecimiento del PIB',
        f'Cambio en PD por cada 1% de aumento en tasa de interés',
        f'Cambio en PD por cada 1% de aumento en inflación',
        f'Cambio en PD por cada punto de aumento en confianza empresarial',
        f'Cambio en PD por cada punto de aumento en volatilidad'
    ]
})
print(coeficientes_df.to_string(index=False))

# Interpretación financiera
print("\n" + "=" * 80)
print("INTERPRETACIÓN FINANCIERA DE LOS COEFICIENTES")
print("=" * 80)
print("\nLos coeficientes indican el cambio esperado en PD (en decimal, 0-1)")
print("por cada unidad de cambio en la variable independiente:")
print("\nEjemplos:")
print(f"  - Si el desempleo aumenta 1 punto porcentual (de 6% a 7%),")
print(f"    la PD aumenta en {modelo_pd.coef_[0]:.4f} ({modelo_pd.coef_[0]*100:.2f} puntos porcentuales)")
print(f"  - Si el crecimiento del PIB aumenta 1 punto porcentual (de 3% a 4%),")
print(f"    la PD cambia en {modelo_pd.coef_[1]:.4f} ({modelo_pd.coef_[1]*100:.2f} puntos porcentuales)")
print(f"  - El intercepto ({modelo_pd.intercept_:.4f}) representa la PD teórica cuando")
print(f"    todas las variables macroeconómicas están en cero")

### Paso 5: Evaluar y Predecir

Evaluamos el modelo en datos de entrenamiento y prueba, luego hacemos predicciones.

In [ ]:
# Paso 5: Evaluar y predecir
y_pred_train = modelo_pd.predict(X_train)
y_pred_test = modelo_pd.predict(X_test)

# Calcular métricas
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)

print("=" * 80)
print("PASO 5: EVALUACIÓN DEL MODELO")
print("=" * 80)

print("\nMétricas de Rendimiento:")
print(f"\n{'Métrica':<25} {'Entrenamiento':<20} {'Prueba':<20}")
print("-" * 65)
print(f"{'R² (Coeficiente Determinación)':<25} {r2_train:<20.4f} {r2_test:<20.4f}")
print(f"{'RMSE (Raíz Error Cuadrático)':<25} {rmse_train:<20.6f} {rmse_test:<20.6f}")
print(f"{'MAE (Error Absoluto Medio)':<25} {mae_train:<20.6f} {mae_test:<20.6f}")

# Interpretación
print("\n" + "=" * 80)
print("INTERPRETACIÓN DE RESULTADOS")
print("=" * 80)
print(f"\nR² = {r2_test:.2%} significa que el modelo explica el {r2_test*100:.2f}%")
print(f"de la variabilidad en la PD usando las variables macroeconómicas.")
print(f"\nRMSE = {rmse_test:.4f} significa que en promedio, nuestras predicciones")
print(f"tienen un error de {rmse_test:.4f} ({rmse_test*100:.2f} puntos porcentuales)")
print(f"en términos de PD.")

# Verificar si hay sobreajuste
diferencia_r2 = r2_train - r2_test
if diferencia_r2 > 0.1:
    print(f"\n⚠ ADVERTENCIA: Posible sobreajuste detectado")
    print(f"  Diferencia R² (train - test): {diferencia_r2:.4f}")
    print(f"  El modelo funciona mejor en entrenamiento que en prueba")
else:
    print(f"\n✓ El modelo generaliza bien (diferencia R²: {diferencia_r2:.4f})")

## Visualización de Resultados

Vamos a visualizar las predicciones vs valores reales:

In [ ]:
# Visualización de resultados
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Gráfico 1: Predicciones vs Valores Reales (Entrenamiento)
axes[0, 0].scatter(y_train, y_pred_train, alpha=0.6, s=50)
min_val = min(y_train.min(), y_pred_train.min())
max_val = max(y_train.max(), y_pred_train.max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Línea perfecta')
axes[0, 0].set_xlabel('PD Real (Entrenamiento)', fontsize=11)
axes[0, 0].set_ylabel('PD Predicha (Entrenamiento)', fontsize=11)
axes[0, 0].set_title(f'Predicciones vs Real - Entrenamiento\n(R² = {r2_train:.3f})', 
                     fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Gráfico 2: Predicciones vs Valores Reales (Prueba)
axes[0, 1].scatter(y_test, y_pred_test, alpha=0.6, s=50, color='green')
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
axes[0, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Línea perfecta')
axes[0, 1].set_xlabel('PD Real (Prueba)', fontsize=11)
axes[0, 1].set_ylabel('PD Predicha (Prueba)', fontsize=11)
axes[0, 1].set_title(f'Predicciones vs Real - Prueba\n(R² = {r2_test:.3f})', 
                     fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Gráfico 3: Serie temporal de predicciones (Entrenamiento)
indices_train = range(len(y_train))
axes[1, 0].plot(indices_train, y_train, 'o-', label='PD Real', linewidth=2, markersize=4, alpha=0.7)
axes[1, 0].plot(indices_train, y_pred_train, 's-', label='PD Predicha', linewidth=2, markersize=4, alpha=0.7)
axes[1, 0].set_xlabel('Observación (Entrenamiento)', fontsize=11)
axes[1, 0].set_ylabel('PD (Decimal)', fontsize=11)
axes[1, 0].set_title('Serie Temporal - Entrenamiento', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Gráfico 4: Serie temporal de predicciones (Prueba)
indices_test = range(len(y_train), len(y_train) + len(y_test))
axes[1, 1].plot(indices_test, y_test, 'o-', label='PD Real', linewidth=2, markersize=4, alpha=0.7, color='green')
axes[1, 1].plot(indices_test, y_pred_test, 's-', label='PD Predicha', linewidth=2, markersize=4, alpha=0.7, color='orange')
axes[1, 1].set_xlabel('Observación (Prueba)', fontsize=11)
axes[1, 1].set_ylabel('PD (Decimal)', fontsize=11)
axes[1, 1].set_title('Serie Temporal - Prueba', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Verificación de Supuestos

Siguiendo la metodología, verificamos los supuestos de la regresión lineal:

In [ ]:
# Verificación de supuestos
residuos_train = y_train - y_pred_train
residuos_test = y_test - y_pred_test

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Residuos vs Valores Predichos (verificar homocedasticidad y linealidad)
axes[0, 0].scatter(y_pred_train, residuos_train, alpha=0.6)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Valores Predichos', fontsize=11)
axes[0, 0].set_ylabel('Residuos', fontsize=11)
axes[0, 0].set_title('Residuos vs Predichos\n(Verificar Homocedasticidad)', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Gráfico 2: Q-Q Plot (verificar normalidad)
probplot(residuos_train, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot de Residuos\n(Verificar Normalidad)', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Gráfico 3: Histograma de residuos
axes[1, 0].hist(residuos_train, bins=15, edgecolor='black', alpha=0.7, density=True)
mu, sigma = residuos_train.mean(), residuos_train.std()
x_norm = np.linspace(residuos_train.min(), residuos_train.max(), 100)
y_norm = stats.norm.pdf(x_norm, mu, sigma)
axes[1, 0].plot(x_norm, y_norm, 'r-', linewidth=2, label='Distribución Normal')
axes[1, 0].set_xlabel('Residuos', fontsize=11)
axes[1, 0].set_ylabel('Densidad', fontsize=11)
axes[1, 0].set_title('Distribución de Residuos', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Gráfico 4: Residuos vs orden temporal (verificar independencia)
axes[1, 1].plot(range(len(residuos_train)), residuos_train, 'o-', alpha=0.6, markersize=3)
axes[1, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Orden de Observación', fontsize=11)
axes[1, 1].set_ylabel('Residuos', fontsize=11)
axes[1, 1].set_title('Residuos vs Orden Temporal\n(Verificar Independencia)', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Pruebas estadísticas
print("=" * 80)
print("VERIFICACIÓN DE SUPUESTOS - PRUEBAS ESTADÍSTICAS")
print("=" * 80)

# Test de normalidad
stat_sw, p_sw = shapiro(residuos_train)
stat_jb, p_jb = jarque_bera(residuos_train)

print(f"\n1. Normalidad de Residuos:")
print(f"   Shapiro-Wilk: estadístico={stat_sw:.4f}, p-valor={p_sw:.4f}")
print(f"   {'✓ Normalidad aceptada' if p_sw > 0.05 else '✗ Normalidad rechazada'} (α=0.05)")
print(f"   Jarque-Bera: estadístico={stat_jb:.4f}, p-valor={p_jb:.4f}")
print(f"   {'✓ Normalidad aceptada' if p_jb > 0.05 else '✗ Normalidad rechazada'} (α=0.05)")

# Homocedasticidad (simplificado)
var_grupo1 = residuos_train[:len(residuos_train)//2].var()
var_grupo2 = residuos_train[len(residuos_train)//2:].var()
ratio_var = max(var_grupo1, var_grupo2) / min(var_grupo1, var_grupo2)

print(f"\n2. Homocedasticidad:")
print(f"   Ratio de varianzas (grupos): {ratio_var:.2f}")
print(f"   {'✓ Homocedasticidad probable' if ratio_var < 2 else '✗ Posible heterocedasticidad'} (ratio < 2)")

# Independencia (correlación serial)
corr_serial = np.corrcoef(residuos_train[:-1], residuos_train[1:])[0, 1]
print(f"\n3. Independencia:")
print(f"   Correlación serial (lag 1): {corr_serial:.4f}")
print(f"   {'✓ Independencia probable' if abs(corr_serial) < 0.2 else '✗ Posible autocorrelación'} (|r| < 0.2)")

print(f"\n4. Media de Residuos: {residuos_train.mean():.6f} (debe ser cercana a 0)")
print(f"   {'✓ Media cercana a cero' if abs(residuos_train.mean()) < 0.001 else '✗ Media no es cero'}")

## Selección de Variables

Aplicamos los métodos de selección de variables para encontrar el modelo óptimo:

In [ ]:
# Función auxiliar para calcular AIC
def calcular_aic_pd(r2, n, p):
    """Calcula AIC para un modelo de regresión"""
    rss = (1 - r2) * n
    if rss <= 0:
        return np.inf
    aic = n * np.log(rss / n) + 2 * (p + 1)
    return aic

def forward_selection_pd(X, y, max_vars=None):
    """Implementa Forward Selection para selección de variables"""
    variables_incluidas = []
    variables_candidatas = list(X.columns)
    mejor_aic = np.inf
    historial = []
    
    # Modelo inicial (solo intercepto)
    modelo_inicial = LinearRegression()
    modelo_inicial.fit(np.ones((len(y), 1)), y)
    y_pred_inicial = modelo_inicial.predict(np.ones((len(y), 1)))
    r2_inicial = r2_score(y, y_pred_inicial)
    aic_inicial = calcular_aic_pd(r2_inicial, len(y), 0)
    mejor_aic = aic_inicial
    
    paso = 1
    while len(variables_candidatas) > 0:
        if max_vars and len(variables_incluidas) >= max_vars:
            break
            
        mejor_mejora = None
        mejor_variable = None
        mejor_r2 = None
        mejor_aic_nuevo = None
        
        for var in variables_candidatas:
            vars_temp = variables_incluidas + [var]
            X_temp = X[vars_temp]
            
            modelo = LinearRegression()
            modelo.fit(X_temp, y)
            y_pred = modelo.predict(X_temp)
            r2 = r2_score(y, y_pred)
            aic_nuevo = calcular_aic_pd(r2, len(y), len(vars_temp))
            
            mejora_aic = mejor_aic - aic_nuevo
            
            if mejor_mejora is None or mejora_aic > mejor_mejora:
                mejor_mejora = mejora_aic
                mejor_variable = var
                mejor_r2 = r2
                mejor_aic_nuevo = aic_nuevo
        
        if mejor_mejora > 0:
            variables_incluidas.append(mejor_variable)
            variables_candidatas.remove(mejor_variable)
            mejor_aic = mejor_aic_nuevo
            historial.append({
                'paso': paso,
                'variable_agregada': mejor_variable,
                'variables': variables_incluidas.copy(),
                'r2': mejor_r2,
                'aic': mejor_aic_nuevo
            })
            paso += 1
        else:
            break
    
    return variables_incluidas, historial

# Aplicar Forward Selection
X_train_df = pd.DataFrame(X_train, columns=variables_independientes)
y_train_series = pd.Series(y_train)

print("=" * 80)
print("SELECCIÓN DE VARIABLES: FORWARD SELECTION")
print("=" * 80)

vars_seleccionadas_pd, historial_pd = forward_selection_pd(X_train_df, y_train_series, max_vars=6)

print(f"\nVariables seleccionadas: {vars_seleccionadas_pd}")
print(f"Número de variables seleccionadas: {len(vars_seleccionadas_pd)}")

In [ ]:
# Comparar modelo completo vs modelo seleccionado
print("\n" + "=" * 80)
print("COMPARACIÓN: MODELO COMPLETO vs MODELO SELECCIONADO")
print("=" * 80)

# Modelo completo
modelo_completo = LinearRegression()
modelo_completo.fit(X_train, y_train)
y_pred_completo = modelo_completo.predict(X_test)
r2_completo = r2_score(y_test, y_pred_completo)
rmse_completo = np.sqrt(mean_squared_error(y_test, y_pred_completo))

# Modelo seleccionado
if len(vars_seleccionadas_pd) > 0:
    indices_vars = [variables_independientes.index(v) for v in vars_seleccionadas_pd]
    X_train_seleccionado = X_train[:, indices_vars]
    X_test_seleccionado = X_test[:, indices_vars]
    
    modelo_seleccionado = LinearRegression()
    modelo_seleccionado.fit(X_train_seleccionado, y_train)
    y_pred_seleccionado = modelo_seleccionado.predict(X_test_seleccionado)
    r2_seleccionado = r2_score(y_test, y_pred_seleccionado)
    rmse_seleccionado = np.sqrt(mean_squared_error(y_test, y_pred_seleccionado))
    
    print(f"\n{'Métrica':<30} {'Modelo Completo':<20} {'Modelo Seleccionado':<20}")
    print("-" * 70)
    print(f"{'Número de Variables':<30} {len(variables_independientes):<20} {len(vars_seleccionadas_pd):<20}")
    print(f"{'R²':<30} {r2_completo:<20.4f} {r2_seleccionado:<20.4f}")
    print(f"{'RMSE':<30} {rmse_completo:<20.6f} {rmse_seleccionado:<20.6f}")
    print(f"\nVariables seleccionadas: {vars_seleccionadas_pd}")
    
    if r2_seleccionado >= r2_completo * 0.95:
        print(f"\n✓ El modelo seleccionado mantiene {r2_seleccionado/r2_completo*100:.1f}% del poder explicativo")
        print(f"  con {len(variables_independientes) - len(vars_seleccionadas_pd)} variables menos (más parsimonioso)")
    else:
        print(f"\n⚠ El modelo seleccionado pierde poder explicativo significativo")
        print(f"  Considerar mantener más variables")
else:
    print("\nNo se seleccionaron variables adicionales")

## Predicciones y Análisis de Escenarios

Finalmente, usamos el modelo para hacer predicciones bajo diferentes escenarios macroeconómicos:

In [ ]:
# Análisis de escenarios
print("=" * 80)
print("ANÁLISIS DE ESCENARIOS: PREDICCIONES DE PD")
print("=" * 80)

# Definir escenarios
escenarios = {
    'Escenario Base': {
        'desempleo': 6.0,
        'pib_crecimiento': 3.0,
        'tasa_interes': 4.0,
        'inflacion': 2.5,
        'confianza_empresarial': 50.0,
        'volatilidad_mercado': 20.0
    },
    'Escenario Optimista': {
        'desempleo': 4.5,
        'pib_crecimiento': 4.5,
        'tasa_interes': 3.0,
        'inflacion': 2.0,
        'confianza_empresarial': 65.0,
        'volatilidad_mercado': 15.0
    },
    'Escenario Pesimista': {
        'desempleo': 8.5,
        'pib_crecimiento': 1.0,
        'tasa_interes': 6.0,
        'inflacion': 4.0,
        'confianza_empresarial': 35.0,
        'volatilidad_mercado': 35.0
    },
    'Escenario Crisis': {
        'desempleo': 10.0,
        'pib_crecimiento': -1.0,
        'tasa_interes': 7.0,
        'inflacion': 5.0,
        'confianza_empresarial': 25.0,
        'volatilidad_mercado': 45.0
    }
}

# Predecir PD para cada escenario
resultados_escenarios = []
for nombre, valores in escenarios.items():
    X_escenario = np.array([[valores[var] for var in variables_independientes]])
    pd_predicha = modelo_pd.predict(X_escenario)[0]
    
    # Asegurar que PD esté en rango [0, 1]
    pd_predicha = np.clip(pd_predicha, 0, 1)
    
    resultados_escenarios.append({
        'Escenario': nombre,
        'PD Predicha (%)': pd_predicha * 100,
        'PD Predicha (Decimal)': pd_predicha,
        **valores
    })

df_escenarios = pd.DataFrame(resultados_escenarios)

print("\nPredicciones de PD bajo diferentes escenarios macroeconómicos:")
print("=" * 80)
print(df_escenarios.to_string(index=False))

In [ ]:
# Visualización de escenarios
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: PD por escenario
axes[0].bar(df_escenarios['Escenario'], df_escenarios['PD Predicha (%)'], 
           color=['#3498db', '#2ecc71', '#e74c3c', '#8e44ad'])
axes[0].set_ylabel('PD Predicha (%)', fontsize=12)
axes[0].set_title('Probabilidad de Incumplimiento por Escenario', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=15)

# Agregar valores en las barras
for i, v in enumerate(df_escenarios['PD Predicha (%)']):
    axes[0].text(i, v + 0.1, f'{v:.2f}%', ha='center', va='bottom', fontweight='bold')

# Gráfico 2: Comparación de variables macro por escenario
variables_vis = ['desempleo', 'pib_crecimiento', 'tasa_interes']
x_pos = np.arange(len(df_escenarios))
width = 0.25

for i, var in enumerate(variables_vis):
    axes[1].bar(x_pos + i*width, df_escenarios[var], width, label=var.replace('_', ' ').title())

axes[1].set_xlabel('Escenario', fontsize=12)
axes[1].set_ylabel('Valor (%)', fontsize=12)
axes[1].set_title('Variables Macroeconómicas por Escenario', fontsize=14, fontweight='bold')
axes[1].set_xticks(x_pos + width)
axes[1].set_xticklabels(df_escenarios['Escenario'], rotation=15)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("INTERPRETACIÓN DE ESCENARIOS")
print("=" * 80)
print("\nEl modelo permite cuantificar el impacto de diferentes condiciones macroeconómicas")
print("en la probabilidad de incumplimiento:")
print(f"\n  • Escenario Base: PD = {df_escenarios.loc[0, 'PD Predicha (%)']:.2f}%")
print(f"  • Escenario Optimista: PD = {df_escenarios.loc[1, 'PD Predicha (%)']:.2f}%")
print(f"    (Reducción de {df_escenarios.loc[0, 'PD Predicha (%)'] - df_escenarios.loc[1, 'PD Predicha (%)']:.2f} puntos porcentuales)")
print(f"  • Escenario Pesimista: PD = {df_escenarios.loc[2, 'PD Predicha (%)']:.2f}%")
print(f"    (Aumento de {df_escenarios.loc[2, 'PD Predicha (%)'] - df_escenarios.loc[0, 'PD Predicha (%)']:.2f} puntos porcentuales)")
print(f"  • Escenario Crisis: PD = {df_escenarios.loc[3, 'PD Predicha (%)']:.2f}%")
print(f"    (Aumento de {df_escenarios.loc[3, 'PD Predicha (%)'] - df_escenarios.loc[0, 'PD Predicha (%)']:.2f} puntos porcentuales)")

## Resumen del Caso Práctico

### Logros del Modelo:

1. **Modelado Exitoso**: Construimos un modelo que explica la PD cuantitativa usando variables macroeconómicas
2. **Metodología Aplicada**: Seguimos los 5 pasos fundamentales de manera sistemática
3. **Verificación de Supuestos**: Validamos que el modelo cumple con los supuestos estadísticos
4. **Selección de Variables**: Identificamos las variables más importantes
5. **Predicciones Útiles**: El modelo permite hacer predicciones bajo diferentes escenarios

### Limitaciones y Consideraciones:

1. **Rango de PD**: La regresión lineal puede predecir valores fuera de [0,1]. En producción, aplicar transformaciones o usar modelos alternativos (Logit transformado)

2. **Estabilidad Temporal**: Los coeficientes pueden cambiar con el tiempo. Re-entrenar periódicamente el modelo

3. **Variables Omitidas**: Puede haber factores no observables que afecten la PD

4. **Causalidad**: El modelo muestra correlación, no necesariamente causalidad

5. **Validación Continua**: Monitorear el rendimiento del modelo en producción y ajustar cuando sea necesario

### Aplicaciones Prácticas:

- **Gestión de Riesgo Crediticio**: Estimar PD para diferentes segmentos de cartera
- **Provisiones**: Calcular provisiones esperadas basadas en escenarios macroeconómicos
- **Estrés Testing**: Evaluar impacto de shocks macroeconómicos en la cartera
- **Pricing**: Ajustar tasas de interés según condiciones macro y PD esperada
- **Regulación**: Cumplir con requerimientos regulatorios (Basel, IFRS 9)

---

**Este ejemplo demuestra cómo aplicar la metodología de regresión lineal a un problema real de gestión de riesgo crediticio, modelando la PD como variable cuantitativa continua.**